# Personalized LOOCV with model-specific SHAP selection

This notebook evaluates SVM, LDA, and XGBoost independently for EEG-only, ET-only, and hybrid EEG+ET data. It implements personalized leave-one-product/epoch-out CV: every test product is predicted using only the other products from the same subject. No SMOTE or other synthetic oversampling is used.

**Leakage boundary:** for every fold, scaling, SHAP feature selection, and classifier fitting use the training products only. The held-out product is transformed and predicted only after those steps are complete.

In [ ]:
# If needed, install once in the active VS Code kernel:
# %pip install numpy pandas scikit-learn xgboost shap

from pathlib import Path
from collections import Counter
import json
import warnings

import numpy as np
import pandas as pd
import shap
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, cohen_kappa_score)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings('ignore', category=UserWarning)

DATA_DIR = Path(r'D:\PT_1\Dataset_Extracted\Separate')
RESULTS_DIR = Path.cwd() / 'personalized_loocv_shap_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TOP_N = 100
RANDOM_STATE = 42
# Set to a list such as ['S01', 'S02'] for a quick pilot; use None for all subjects.
SUBJECTS_TO_RUN = None

print(f'Data: {DATA_DIR}')
print(f'Outputs: {RESULTS_DIR}')

In [ ]:
# Load subject-wise final features. This is loading only: it does not alter
# synchronization, feature order, labels, or the supplied 7-IMF representation.

def discover_subjects(data_dir):
    subject_file = data_dir / 'subjects.txt'
    if subject_file.exists():
        return [line.strip() for line in subject_file.read_text().splitlines() if line.strip()]
    return sorted(path.name.replace('_eeg_X.npy', '') for path in data_dir.glob('*_eeg_X.npy'))

def load_subjects(data_dir, selected_subjects=None):
    subjects = discover_subjects(data_dir)
    if selected_subjects is not None:
        requested = set(selected_subjects)
        subjects = [sid for sid in subjects if sid in requested]

    data = {}
    for sid in subjects:
        eeg = np.load(data_dir / f'{sid}_eeg_X.npy').astype(np.float64)
        et = np.load(data_dir / f'{sid}_et_X.npy').astype(np.float64)
        y = np.load(data_dir / f'{sid}_y.npy').astype(int).ravel()
        if not (len(eeg) == len(et) == len(y)):
            raise ValueError(f'{sid}: EEG, ET, and y must have identical row counts.')
        data[sid] = {'eeg': eeg, 'et': et, 'y': y}
    return data

subject_data = load_subjects(DATA_DIR, SUBJECTS_TO_RUN)
if not subject_data:
    raise RuntimeError('No subject files were found.')

# This pooled audit index preserves subject identity. It is never given to a classifier.
pooled_subject_id = np.concatenate([np.repeat(sid, len(item['y'])) for sid, item in subject_data.items()])
pooled_y = np.concatenate([item['y'] for item in subject_data.values()])

summary = pd.DataFrame([
    {'subject_id': sid, 'n_products': len(item['y']), 'buy': int(item['y'].sum()),
     'no_buy': int((item['y'] == 0).sum()), 'eeg_columns': item['eeg'].shape[1],
     'et_columns': item['et'].shape[1]}
    for sid, item in subject_data.items()
])
display(summary)
print(f'{len(summary)} subjects; {len(pooled_y)} product windows.')

In [ ]:
# Build stable, human-readable feature metadata. These feature IDs are retained
# across EEG-only and hybrid experiments, enabling meaningful intersections.

CHANNELS = ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2',
            'F7', 'F8', 'T7', 'T8', 'P7', 'P8', 'Fz', 'Cz', 'Pz']

eeg_config = json.loads((DATA_DIR / 'feature_names_eeg.json').read_text())
et_config = json.loads((DATA_DIR / 'feature_names_et.json').read_text())
EEG_FEATURES = eeg_config['features']
ET_FEATURES = et_config['features']

def build_eeg_metadata(n_columns):
    rows = []
    per_imf = len(EEG_FEATURES)
    imfs_per_channel = n_columns // (len(CHANNELS) * per_imf)
    for column in range(n_columns):
        channel_index = column // (imfs_per_channel * per_imf)
        remainder = column % (imfs_per_channel * per_imf)
        imf_index = remainder // per_imf
        feature_index = remainder % per_imf
        feature_name = EEG_FEATURES[feature_index]
        channel = CHANNELS[channel_index]
        rows.append({'feature_id': f'EEG|{channel}|IMF{imf_index + 1}|{feature_name}',
                     'source_modality': 'EEG', 'source_column': column,
                     'channel': channel, 'imf': imf_index + 1,
                     'feature_name': feature_name})
    return pd.DataFrame(rows)

def build_et_metadata(n_columns):
    return pd.DataFrame([
        {'feature_id': f'ET|{ET_FEATURES[i]}', 'source_modality': 'ET',
         'source_column': i, 'channel': None, 'imf': None, 'feature_name': ET_FEATURES[i]}
        for i in range(n_columns)
    ])

n_eeg = next(iter(subject_data.values()))['eeg'].shape[1]
n_et = next(iter(subject_data.values()))['et'].shape[1]
metadata_by_modality = {
    'EEG': build_eeg_metadata(n_eeg),
    'ET': build_et_metadata(n_et),
}
metadata_by_modality['Hybrid'] = pd.concat(
    [metadata_by_modality['EEG'], metadata_by_modality['ET']], ignore_index=True
)
for metadata in metadata_by_modality.values():
    metadata['model_column'] = np.arange(len(metadata))

display(metadata_by_modality['Hybrid'].head())

In [ ]:
# Classifier factories and SHAP-based selection. The SHAP selector is the same
# classifier family as the eventual evaluator, but is fit only on that fold's
# scaled training products. No feature-selection fallback is used.

def make_model(model_name):
    if model_name == 'SVM':
        # Linear SVM mirrors the base paper and permits exact linear SHAP values.
        return SVC(kernel='linear', C=1.0, probability=False, random_state=RANDOM_STATE)
    if model_name == 'LDA':
        # Shrinkage keeps LDA numerically stable when selected features exceed samples.
        return LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
    if model_name == 'XGB':
        return XGBClassifier(
            n_estimators=150, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            objective='binary:logistic', eval_metric='logloss',
            random_state=RANDOM_STATE, n_jobs=1, verbosity=0
        )
    raise ValueError(f'Unknown model: {model_name}')

def positive_score(model, X):
    if hasattr(model, 'predict_proba'):
        class_index = list(model.classes_).index(1)
        return model.predict_proba(X)[:, class_index]
    return model.decision_function(X)

def normalise_shap_values(values):
    if isinstance(values, list):
        values = values[-1]
    if hasattr(values, 'values'):
        values = values.values
    values = np.asarray(values)
    if values.ndim == 3:  # SHAP versions may put the class axis last.
        values = values[:, :, -1]
    if values.ndim != 2:
        raise ValueError(f'Unexpected SHAP array shape: {values.shape}')
    return values

def select_top_features_with_shap(model_name, X_train_scaled, y_train, top_n):
    selector = make_model(model_name)
    selector.fit(X_train_scaled, y_train)

    if model_name == 'XGB':
        shap_values = shap.TreeExplainer(selector).shap_values(X_train_scaled)
    else:
        shap_values = shap.LinearExplainer(selector, X_train_scaled).shap_values(X_train_scaled)

    importance = np.mean(np.abs(normalise_shap_values(shap_values)), axis=0)
    selected = np.argsort(importance)[::-1][:min(top_n, X_train_scaled.shape[1])]
    return selected.astype(int), importance[selected], selector

In [ ]:
# One personalized LOOCV run. Test rows are never included in scaler fitting,
# SHAP calculation, selected-feature determination, or classifier fitting.

MODELS = ['SVM', 'LDA', 'XGB']
MODALITIES = ['EEG', 'ET', 'Hybrid']

def get_modality_matrix(subject, modality):
    if modality == 'EEG':
        return subject['eeg']
    if modality == 'ET':
        return subject['et']
    if modality == 'Hybrid':
        return np.column_stack([subject['eeg'], subject['et']])
    raise ValueError(modality)

def run_personalized_loocv(subject_data):
    prediction_rows = []
    selection_rows = []

    for subject_number, (sid, subject) in enumerate(subject_data.items(), start=1):
        y = subject['y']
        print(f'[{subject_number}/{len(subject_data)}] {sid}: {len(y)} product windows')

        for modality in MODALITIES:
            X = get_modality_matrix(subject, modality)
            feature_metadata = metadata_by_modality[modality]
            n_select = min(TOP_N, X.shape[1])

            for held_out_index in range(len(y)):
                train_mask = np.ones(len(y), dtype=bool)
                train_mask[held_out_index] = False
                X_train_raw, X_test_raw = X[train_mask], X[~train_mask]
                y_train, y_test = y[train_mask], y[~train_mask]

                # Scaling precedes SHAP and is fit exclusively on training products.
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train_raw)
                X_test_scaled = scaler.transform(X_test_raw)

                for model_name in MODELS:
                    if np.unique(y_train).size < 2:
                        # A one-class training fold cannot support SHAP or a binary classifier.
                        majority = int(y_train[0])
                        prediction_rows.append({
                            'subject_id': sid, 'held_out_product_index': held_out_index,
                            'modality': modality, 'model': model_name,
                            'y_true': int(y_test[0]), 'y_pred': majority, 'score': np.nan,
                            'status': 'single_class_training_fold'
                        })
                        continue

                    selected, selected_importance, _ = select_top_features_with_shap(
                        model_name, X_train_scaled, y_train, n_select
                    )
                    selected_metadata = feature_metadata.iloc[selected]
                    selection_rows.append({
                        'subject_id': sid, 'held_out_product_index': held_out_index,
                        'modality': modality, 'model': model_name,
                        'n_selected': len(selected),
                        'feature_ids_json': json.dumps(selected_metadata['feature_id'].tolist()),
                        'model_columns_json': json.dumps(selected.tolist()),
                        'mean_abs_shap_json': json.dumps([float(x) for x in selected_importance])
                    })

                    evaluator = make_model(model_name)
                    evaluator.fit(X_train_scaled[:, selected], y_train)
                    y_pred = int(evaluator.predict(X_test_scaled[:, selected])[0])
                    score = float(positive_score(evaluator, X_test_scaled[:, selected])[0])
                    prediction_rows.append({
                        'subject_id': sid, 'held_out_product_index': held_out_index,
                        'modality': modality, 'model': model_name,
                        'y_true': int(y_test[0]), 'y_pred': y_pred, 'score': score,
                        'status': 'ok'
                    })

    return pd.DataFrame(prediction_rows), pd.DataFrame(selection_rows)

In [ ]:
# Run all 9 experiments (3 models x 3 modalities). This is computationally
# intensive because SHAP is refit legitimately for every held-out product.
predictions, selected_feature_lists = run_personalized_loocv(subject_data)

predictions.to_csv(RESULTS_DIR / 'fold_predictions.csv', index=False)
selected_feature_lists.to_csv(RESULTS_DIR / 'fold_selected_feature_lists.csv', index=False)
print('Saved fold-level predictions and model-specific SHAP feature lists.')

In [ ]:
# Per-subject metrics and averages across subjects. AUC is undefined for a
# subject with a single observed class and is recorded as NaN in that case.

def metric_row(frame):
    y_true = frame['y_true'].to_numpy()
    y_pred = frame['y_pred'].to_numpy()
    usable_scores = frame['score'].notna()
    auc = np.nan
    if len(np.unique(y_true)) == 2 and usable_scores.all():
        auc = roc_auc_score(y_true, frame['score'].to_numpy())
    return {
        'n_products': len(frame),
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'auc_roc': auc,
        'cohen_kappa': cohen_kappa_score(y_true, y_pred),
        'nonstandard_folds': int((frame['status'] != 'ok').sum())
    }

per_subject_metric_rows = []
for (subject_id, modality, model), frame in predictions.groupby(['subject_id', 'modality', 'model'], sort=True):
    row = metric_row(frame)
    row.update({'subject_id': subject_id, 'modality': modality, 'model': model})
    per_subject_metric_rows.append(row)
per_subject_metrics = pd.DataFrame(per_subject_metric_rows)
average_metrics = (
    per_subject_metrics.groupby(['modality', 'model'], sort=True)[
        ['accuracy', 'precision', 'recall', 'f1', 'auc_roc', 'cohen_kappa']
    ].mean().reset_index()
)

per_subject_metrics.to_csv(RESULTS_DIR / 'per_subject_metrics.csv', index=False)
average_metrics.to_csv(RESULTS_DIR / 'average_subject_metrics.csv', index=False)
display(average_metrics.sort_values(['model', 'modality']))

In [ ]:
# Feature commonality analysis. 'Within model' means features present in every
# successful LOOCV selection set for a given model and modality. 'Across models'
# means the intersection of SVM, LDA, and XGB within-model common feature sets
# for the same modality. Frequencies are also exported because strict
# intersections can legitimately be empty.

def metadata_lookup(modality):
    return metadata_by_modality[modality].set_index('feature_id')

frequency_rows = []
common_within_rows = []
common_sets = {}

for (modality, model), group in selected_feature_lists.groupby(['modality', 'model'], sort=True):
    fold_sets = [set(json.loads(value)) for value in group['feature_ids_json']]
    common_set = set.intersection(*fold_sets) if fold_sets else set()
    common_sets[(modality, model)] = common_set
    counts = Counter(feature_id for fold_set in fold_sets for feature_id in fold_set)
    lookup = metadata_lookup(modality)

    for feature_id, count in counts.items():
        row = lookup.loc[feature_id].to_dict()
        row.update({'modality': modality, 'model': model, 'selection_count': count,
                    'n_successful_folds': len(fold_sets),
                    'selection_rate': count / len(fold_sets)})
        frequency_rows.append(row)

    for feature_id in sorted(common_set):
        row = lookup.loc[feature_id].to_dict()
        row.update({'modality': modality, 'model': model,
                    'n_folds_intersected': len(fold_sets)})
        common_within_rows.append(row)

selection_frequency = pd.DataFrame(frequency_rows).sort_values(
    ['modality', 'model', 'selection_rate', 'feature_id'], ascending=[True, True, False, True]
)
common_within_model = pd.DataFrame(common_within_rows)

common_across_rows = []
for modality in MODALITIES:
    shared = set.intersection(*(common_sets.get((modality, model), set()) for model in MODELS))
    lookup = metadata_lookup(modality)
    for feature_id in sorted(shared):
        row = lookup.loc[feature_id].to_dict()
        row.update({'modality': modality, 'models_intersected': '|'.join(MODELS)})
        common_across_rows.append(row)
common_across_models = pd.DataFrame(common_across_rows)

selection_frequency.to_csv(RESULTS_DIR / 'feature_selection_frequency.csv', index=False)
common_within_model.to_csv(RESULTS_DIR / 'common_features_within_model.csv', index=False)
common_across_models.to_csv(RESULTS_DIR / 'common_features_across_models.csv', index=False)

print('Common-feature exports saved.')
display(common_within_model)
display(common_across_models)

## Output files

- `fold_predictions.csv`: held-out product predictions for every classifier and modality.
- `fold_selected_feature_lists.csv`: one model-specific SHAP top-feature list per successful fold.
- `per_subject_metrics.csv`: personalized evaluation metrics for every subject, classifier, and modality.
- `average_subject_metrics.csv`: mean per-subject metrics across the study population.
- `common_features_within_model.csv`: strict intersections of selected features within a model and modality.
- `common_features_across_models.csv`: strict intersections across SVM, LDA, and XGB for each modality.
- `feature_selection_frequency.csv`: ranked selection frequencies, useful when strict intersections are empty.
